In [4]:
!pip install drain3 pandas numpy matplotlib


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import os
import csv
# load log file và đếm tổng số dòng
log_file_path = 'HDFS_2k.log'

with open(log_file_path, 'r', encoding='utf-8') as f:
    log_lines = f.readlines()

total_lines = len(log_lines)
print(f"Tổng số dòng trong log file: {total_lines}")


Tổng số dòng trong log file: 2000


In [6]:
# config những cái của drain3
config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
miner = TemplateMiner(config=config)

# Parse log
for idx, line in enumerate(log_lines):
    line = line.strip()
    if line:
        result = miner.add_log_message(line)
print(f"Đã parse xong {total_lines} dòng log.")   

Đã parse xong 2000 dòng log.


In [7]:

# Liệt kê template và đếm số dòng của mỗi template ( đang sắp xếp theo số lần xuất hiện )
sorted_clusters = sorted(miner.drain.clusters, key=lambda c: c.size, reverse=True)
for cluster in sorted_clusters:
    print(f"  [{cluster.cluster_id}] (count={cluster.size}): {cluster.get_template()}")


  [2] (count=314): <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
  [1] (count=311): <*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating
  [3] (count=292): <*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
  [4] (count=292): <*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
  [7] (count=263): <*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
  [10] (count=224): <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>
  [5] (count=115): <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
  [8] (count=80): <*> <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
  [9] (count=80): <*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> exception while serving <*> to <*>
  [6] (count=20): <*> <*> 13 INFO dfs.DataBlockScanner: Verification

In [8]:
# Tạo folder results và nhét file top_templates.csv vào trong đó
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)
csv_file_path = os.path.join(output_dir, 'top_templates.csv')

# chọn top 10 và ghi vào csv
top_10_clusters = sorted_clusters[:10]
with open(csv_file_path, mode='w', encoding='utf-8', newline='') as csv_file:
    fieldnames = ['template_id', 'template', 'count']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    # Viết dòng Header (Tiêu đề cột)
    writer.writeheader()

    # Viết từng dòng dữ liệu
    for cluster in top_10_clusters:
        writer.writerow({
            'template_id': cluster.cluster_id,
            'template': cluster.get_template(),
            'count': cluster.size
        })
print(f" Đã xuất thành công ")

 Đã xuất thành công 


In [9]:
sim_th_values = [0.3, 0.5, 0.7]
tuning_results = {}

# Vòng lặp chạy qua từng giá trị
for sim_th in sim_th_values:
    print(f"trường hợp sim_th = {sim_th}:")
    config = TemplateMinerConfig()
    config.drain_sim_th = sim_th
    config.drain_depth = 4

    miner = TemplateMiner(config=config)

    for line in log_lines:
        line = line.strip()
        if line:
            miner.add_log_message(line)
    num_templates = len(miner.drain.clusters)
    tuning_results[sim_th] = num_templates
    print(f"số lượng template sinh ra trong trường hợp này: {num_templates}\n")


trường hợp sim_th = 0.3:
số lượng template sinh ra trong trường hợp này: 17

trường hợp sim_th = 0.5:
số lượng template sinh ra trong trường hợp này: 21

trường hợp sim_th = 0.7:
số lượng template sinh ra trong trường hợp này: 820



In [ ]:
    import re
    from datetime import datetime
    import pandas as pd

    # Tạo time series
    def create_template_timeseries(log_entries, window='5min'):
        """
        Biến parsed log thành time series per template.
        
        Args:
            log_entries: list of (timestamp, template_id) tuples
            window: aggregation window
        
        Returns:
            DataFrame: columns = template IDs, rows = time windows, values = count
        """
        df = pd.DataFrame(log_entries, columns=['timestamp', 'template_id'])
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Group by time window + template → count
        ts = df.groupby([pd.Grouper(key='timestamp', freq=window), 'template_id']).size()
        ts = ts.unstack(fill_value=0)  # pivot: rows=time, cols=template
        return ts

    # Nạp data vào hàm 
    log_entries = []

    # Regex để bắt timestamp HDFS: yymmdd hhmmss
    hdfs_time_pattern = re.compile(r'^(\d{6} \d{6})')

    print("Đang trích xuất timestamp và template_id từ log...")

    for line in log_lines:
        line = line.strip()
        if not line:
            continue
        
        # Match template bằng miner đã train ở Phase 1
        match_result = miner.match(line)
        
        if match_result:
            template_id = f"T-{match_result.cluster_id:03d}"
        else:
            template_id = "T-Unknown"
        
        # Trích xuất timestamp
        time_match = hdfs_time_pattern.search(line)
        if time_match:
            time_str = time_match.group(1)
            timestamp = datetime.strptime(time_str, '%y%m%d %H%M%S')
            log_entries.append((timestamp, template_id))

    print(f"✓ Đã chuẩn bị {len(log_entries)} log entries hợp lệ.\n")

    # Tạo time series với window 4 phút
    time_series_df = create_template_timeseries(log_entries, window='5min')

    print(f"\n✓ Hoàn thành! Time series shape: {time_series_df.shape}")
    print(f"  - Số time windows: {time_series_df.shape[0]}")
    print(f"  - Số templates: {time_series_df.shape[1]}")
    print(f"\n{'='*80}")
    print("MẪU TIME SERIES (5 dòng đầu):")
    print(time_series_df.head())
    print(f"{'='*80}")

    # Hiển thị thống kê cơ bản
    print("\nTHỐNG KÊ TEMPLATE COUNT:")
    print(time_series_df.sum().sort_values(ascending=False).head(10))


Đang trích xuất timestamp và template_id từ log...
✓ Đã chuẩn bị 2000 log entries hợp lệ.

Đang tạo template count time series (window=5min)...

✓ Hoàn thành! Time series shape: (305, 820)
  - Số time windows: 305
  - Số templates: 820

MẪU TIME SERIES (5 dòng đầu):
template_id          T-001  T-002  T-003  T-004  T-005  T-006  T-007  T-008  \
timestamp                                                                     
2008-11-09 20:35:00      1      1      0      0      0      0      0      0   
2008-11-09 20:40:00      0      0      4      2      0      0      0      0   
2008-11-09 20:45:00      0      0      1      1      1      1      1      1   
2008-11-09 20:50:00      1      0      1      0      0      0      0      0   
2008-11-09 20:55:00      0      0      3      0      0      0      0      0   

template_id          T-009  T-010  ...  T-811  T-812  T-813  T-814  T-815  \
timestamp                          ...                                      
2008-11-09 20:35:00      

In [ ]:
# apply anomaly detector 

import numpy as np
from sklearn.ensemble import IsolationForest



print("="*80)
print("PHƯƠNG PHÁP 1: 3O")
print("="*80)

# Tính mean và std cho mỗi template
template_stats = pd.DataFrame({
    'mean': time_series_df.mean(),
    'std': time_series_df.std(),
    'max': time_series_df.max()
})

# Phát hiện anomaly: count > mean + 3*std
anomalies_3sigma = {}

for col in time_series_df.columns:
    mean_val = template_stats.loc[col, 'mean']
    std_val = template_stats.loc[col, 'std']
    threshold = mean_val + 3 * std_val
    
    # Tìm các time window có count vượt threshold
    anomaly_mask = time_series_df[col] > threshold
    anomaly_windows = time_series_df[anomaly_mask]
    
    if len(anomaly_windows) > 0:
        anomalies_3sigma[col] = {
            'count': len(anomaly_windows),
            'threshold': threshold,
            'max_value': template_stats.loc[col, 'max']
        }

print(f"\n✓ Phát hiện {len(anomalies_3sigma)} templates có spike bất thường:\n")
for template_id, info in sorted(anomalies_3sigma.items(), key=lambda x: x[1]['count'], reverse=True)[:10]:
    print(f"{template_id}: {info['count']} windows (threshold: {info['threshold']:.2f}, max: {info['max_value']})")



print("\n" + "="*80)
print("PHƯƠNG PHÁP 2: ISOLATION FOREST")
print("="*80)

# Train Isolation Forest
iso_forest = IsolationForest(contamination=0.1, random_state=42)
predictions = iso_forest.fit_predict(time_series_df)

# -1 = anomaly, 1 = normal
anomaly_indices = np.where(predictions == -1)[0]

print(f"\n✓ Phát hiện {len(anomaly_indices)} anomaly windows")
print(f"\nTop 10 anomaly windows:")

for idx in anomaly_indices[:10]:
    timestamp = time_series_df.index[idx]
    total_logs = time_series_df.iloc[idx].sum()
    top_template = time_series_df.iloc[idx].idxmax()
    max_count = time_series_df.iloc[idx].max()
    
    print(f"  {timestamp}: {total_logs} logs (top: {top_template}={max_count})")

print("\n" + "="*80)


PHƯƠNG PHÁP 1: 3O

✓ Phát hiện 820 templates có spike bất thường:

T-114: 15 windows (threshold: 0.70, max: 1)
T-101: 12 windows (threshold: 1.56, max: 3)
T-532: 12 windows (threshold: 1.79, max: 3)
T-528: 11 windows (threshold: 1.60, max: 3)
T-531: 10 windows (threshold: 1.54, max: 2)
T-002: 9 windows (threshold: 0.69, max: 2)
T-001: 8 windows (threshold: 0.67, max: 2)
T-003: 8 windows (threshold: 1.41, max: 4)
T-105: 8 windows (threshold: 1.44, max: 2)
T-103: 6 windows (threshold: 1.31, max: 3)

PHƯƠNG PHÁP 2: ISOLATION FOREST

✓ Phát hiện 31 anomaly windows

Top 10 anomaly windows:
  2008-11-09 23:55:00: 7 logs (top: T-003=2)
  2008-11-10 01:20:00: 5 logs (top: T-101=1)
  2008-11-10 01:35:00: 10 logs (top: T-100=1)
  2008-11-10 01:55:00: 12 logs (top: T-100=3)
  2008-11-10 08:05:00: 8 logs (top: T-197=1)
  2008-11-10 10:40:00: 10 logs (top: T-105=2)
  2008-11-10 11:00:00: 9 logs (top: T-100=2)
  2008-11-10 11:10:00: 10 logs (top: T-100=3)
  2008-11-10 12:20:00: 7 logs (top: T-100=2)

In [ ]:
# Template nào spike bất thường? (từ kết quả 3-sigma)
if len(anomalies_3sigma) > 0:
    print(f"\n✓ {len(anomalies_3sigma)} templates có spike bất thường:\n")
    
    for template_id, info in sorted(anomalies_3sigma.items(), key=lambda x: x[1]['max_value'], reverse=True):
        mean_val = template_stats.loc[template_id, 'mean']
        max_val = info['max_value']
        spike_ratio = max_val / mean_val if mean_val > 0 else float('inf')
        
        print(f"{template_id}:")
        print(f"  - Max spike: {max_val} logs (trung bình: {mean_val:.2f})")
        print(f"  - Tỷ lệ spike: {spike_ratio:.2f}x so với trung bình")
        print(f"  - Số lần spike: {info['count']} windows")
else:
    print("\n✗ Không phát hiện template nào có spike bất thường (theo 3-sigma)")




✓ 820 templates có spike bất thường:

T-169:
  - Max spike: 64 logs (trung bình: 0.40)
  - Tỷ lệ spike: 160.00x so với trung bình
  - Số lần spike: 3 windows
T-234:
  - Max spike: 39 logs (trung bình: 0.40)
  - Tỷ lệ spike: 97.50x so với trung bình
  - Số lần spike: 5 windows
T-511:
  - Max spike: 29 logs (trung bình: 0.37)
  - Tỷ lệ spike: 78.27x so với trung bình
  - Số lần spike: 5 windows
T-510:
  - Max spike: 25 logs (trung bình: 0.33)
  - Tỷ lệ spike: 74.75x so với trung bình
  - Số lần spike: 5 windows
T-518:
  - Max spike: 7 logs (trung bình: 0.45)
  - Tỷ lệ spike: 15.47x so với trung bình
  - Số lần spike: 4 windows
T-236:
  - Max spike: 6 logs (trung bình: 0.04)
  - Tỷ lệ spike: 166.36x so với trung bình
  - Số lần spike: 2 windows
T-100:
  - Max spike: 5 logs (trung bình: 0.42)
  - Tỷ lệ spike: 11.82x so với trung bình
  - Số lần spike: 4 windows
T-003:
  - Max spike: 4 logs (trung bình: 0.09)
  - Tỷ lệ spike: 45.19x so với trung bình
  - Số lần spike: 8 windows
T-101:
  - 

In [ ]:
known_templates = set()
new_template_events = []
hdfs_time_pattern = re.compile(r'^(\d{6} \d{6})')

print("\nĐang phân tích log theo thứ tự thời gian...\n")

for line in log_lines:
    line = line.strip()
    if not line:
        continue
    
    # Parse và match template
    result = miner.match(line)
    if not result:
        continue
    
    template = result.get_template()
    template_id = f"T-{result.cluster_id:03d}"
    
    # Trích xuất timestamp
    time_match = hdfs_time_pattern.search(line)
    if not time_match:
        continue
    
    time_str = time_match.group(1)
    timestamp = datetime.strptime(time_str, '%y%m%d %H%M%S')
    
    # Detect template mới
    if template not in known_templates:
        known_templates.add(template)
        new_template_events.append({
            'timestamp': timestamp,
            'template_id': template_id,
            'template': template
        })
# Hiển thị tóm tắt
print("\nTimeline template mới xuất hiện:\n")
for event in new_template_events:
    print(f"{event['timestamp']} - {event['template_id']}")

print("\n" + "="*80)


Đang phân tích log theo thứ tự thời gian...


Timeline template mới xuất hiện:

2008-11-09 20:36:15 - T-001
2008-11-09 20:38:07 - T-002
2008-11-09 20:40:05 - T-003
2008-11-09 20:40:15 - T-004
2008-11-09 20:46:55 - T-005
2008-11-09 20:47:22 - T-006
2008-11-09 20:48:15 - T-007
2008-11-09 20:48:42 - T-008
2008-11-09 20:49:25 - T-009
2008-11-09 20:50:35 - T-010
2008-11-09 20:51:57 - T-011
2008-11-09 20:53:15 - T-012
2008-11-09 20:54:12 - T-013
2008-11-09 20:57:42 - T-014
2008-11-09 20:57:49 - T-015
2008-11-09 20:57:54 - T-016
2008-11-09 20:58:58 - T-017
2008-11-09 20:59:31 - T-018
2008-11-09 21:00:22 - T-019
2008-11-09 21:00:37 - T-020
2008-11-09 21:02:48 - T-021
2008-11-09 21:04:58 - T-022
2008-11-09 21:06:37 - T-023
2008-11-09 21:06:56 - T-024
2008-11-09 21:07:12 - T-025
2008-11-09 21:08:01 - T-026
2008-11-09 21:08:07 - T-027
2008-11-09 21:08:12 - T-028
2008-11-09 21:09:21 - T-029
2008-11-09 21:09:35 - T-030
2008-11-09 21:12:16 - T-031
2008-11-09 21:14:03 - T-032
2008-11-09 21:14:53 - T

In [21]:

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("="*80)
print("TÍNH PRECISION/RECALL CHO LOG-BASED ANOMALY DETECTION")
print("="*80)

# Kiểm tra file label
label_files = ['anomaly_label.csv', 'HDFS_label.csv', 'labels.csv']
label_file = None

for f in label_files:
    if os.path.exists(f):
        label_file = f
        break

if label_file:
    print(f"\n✓ Tìm thấy file label: {label_file}")
    
    # Load labels
    labels_df = pd.read_csv(label_file)
    print(f"\nCấu trúc file label:")
    print(labels_df.head())
    print(f"\nCác cột: {labels_df.columns.tolist()}")
    
    # Map labels với time windows của time_series_df
    # (Cần điều chỉnh tùy theo cấu trúc file label)
    
else:
    print("\n✗ Không tìm thấy file label")
    print("\nVới HDFS dataset, file label thường có dạng:")
    print("  - BlockId, Label")
    print("  - hoặc: Timestamp, Label")
    print("  (Label: 0=Normal, 1=Anomaly)")
    
    print("\n" + "-"*80)
    print("DEMO: Tính precision/recall với predictions từ Isolation Forest")
    print("-"*80)
    
    # Giả lập: Tạo ground truth labels
    # Trong thực tế, bạn cần map từ file label thật
    np.random.seed(42)
    y_true = np.random.choice([0, 1], size=len(time_series_df), p=[0.85, 0.15])
    
    # Predictions từ Isolation Forest
    # -1 (anomaly) -> 1, 1 (normal) -> 0
    y_pred = (predictions == -1).astype(int)
    
    # Tính metrics
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"\n{'Metric':<20} {'Value'}")
    print("-"*40)
    print(f"{'Precision':<20} {precision:.4f}")
    print(f"{'Recall':<20} {recall:.4f}")
    print(f"{'F1-Score':<20} {f1:.4f}")
    
    print(f"\n{'Confusion Matrix:'}")
    print(f"\n                 Predicted")
    print(f"                 Normal    Anomaly")
    print(f"Actual Normal    {cm[0,0]:<9} {cm[0,1]:<9}")
    print(f"       Anomaly   {cm[1,0]:<9} {cm[1,1]:<9}")
    
    # Giải thích metrics
    TP = cm[1,1]  # True Positive
    FP = cm[0,1]  # False Positive
    FN = cm[1,0]  # False Negative
    TN = cm[0,0]  # True Negative
    
    print(f"\n{'Giải thích:'}")
    print(f"  - True Positives (TP):  {TP} (đúng là anomaly)")
    print(f"  - False Positives (FP): {FP} (nhầm normal thành anomaly)")
    print(f"  - False Negatives (FN): {FN} (bỏ sót anomaly)")
    print(f"  - True Negatives (TN):  {TN} (đúng là normal)")
    
    print(f"\n{'Ý nghĩa:'}")
    print(f"  - Precision = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision:.4f}")
    print(f"    → Trong số dự đoán là anomaly, {precision*100:.1f}% đúng")
    print(f"  - Recall = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.4f}")
    print(f"    → Phát hiện được {recall*100:.1f}% anomaly thực tế")

print("\n" + "="*80)
print("LƯU Ý: Đây là demo với synthetic labels")
print("Để tính chính xác, cần file label thật từ HDFS dataset!")
print("="*80)


TÍNH PRECISION/RECALL CHO LOG-BASED ANOMALY DETECTION

✗ Không tìm thấy file label

Với HDFS dataset, file label thường có dạng:
  - BlockId, Label
  - hoặc: Timestamp, Label
  (Label: 0=Normal, 1=Anomaly)

--------------------------------------------------------------------------------
DEMO: Tính precision/recall với predictions từ Isolation Forest
--------------------------------------------------------------------------------

Metric               Value
----------------------------------------
Precision            0.2581
Recall               0.1667
F1-Score             0.2025

Confusion Matrix:

                 Predicted
                 Normal    Anomaly
Actual Normal    234       23       
       Anomaly   40        8        

Giải thích:
  - True Positives (TP):  8 (đúng là anomaly)
  - False Positives (FP): 23 (nhầm normal thành anomaly)
  - False Negatives (FN): 40 (bỏ sót anomaly)
  - True Negatives (TN):  234 (đúng là normal)

Ý nghĩa:
  - Precision = TP/(TP+FP) = 8/(8+23) =

In [25]:
# ==========================================
# PHASE 3: EMBEDDING + CROSS-SIGNAL
# ==========================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("PHASE 3: TF-IDF EMBEDDING & TEMPLATE CLUSTERING")
print("="*80)

# ==========================================
# 3.1: TF-IDF trên templates
# ==========================================

print("\nBước 1: Tạo TF-IDF vectors cho templates...")

# Lấy tất cả templates từ Drain3
templates = []
template_ids = []

for cluster in miner.drain.clusters:
    template = cluster.get_template()
    templates.append(template)
    template_ids.append(f"T-{cluster.cluster_id:03d}")

print(f"  - Tổng số templates: {len(templates)}")

# Tạo TF-IDF vectors
vectorizer = TfidfVectorizer(
    analyzer='word',
    token_pattern=r'\S+',  # Tách theo space
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(templates)
print(f"  - TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  - Vocabulary size: {len(vectorizer.vocabulary_)}")

# ==========================================
# 3.2: Tính similarity matrix
# ==========================================

print("\nBước 2: Tính cosine similarity matrix...")

similarity_matrix = cosine_similarity(tfidf_matrix)
print(f"  - Similarity matrix shape: {similarity_matrix.shape}")
print(f"  - Mean similarity: {similarity_matrix.mean():.4f}")
print(f"  - Max similarity (non-diagonal): {np.max(similarity_matrix - np.eye(len(templates))):.4f}")

# Hiển thị top similar template pairs
print("\nTop 5 cặp templates giống nhau nhất:")
similarity_pairs = []
for i in range(len(templates)):
    for j in range(i+1, len(templates)):
        similarity_pairs.append((template_ids[i], template_ids[j], similarity_matrix[i, j]))

similarity_pairs.sort(key=lambda x: x[2], reverse=True)
for tid1, tid2, sim in similarity_pairs[:5]:
    print(f"  {tid1} <-> {tid2}: {sim:.4f}")

# ==========================================
# 3.3: Template clustering
# ==========================================

print("\nBước 3: Clustering templates...")

# Sử dụng Agglomerative Clustering
n_clusters = min(5, len(templates))  # Tối đa 5 clusters
clustering = AgglomerativeClustering(
    n_clusters=n_clusters,
    metric='precomputed',
    linkage='average'
)

# Convert similarity to distance
distance_matrix = 1 - similarity_matrix
cluster_labels = clustering.fit_predict(distance_matrix)

print(f"  - Số clusters: {n_clusters}")

# Hiển thị TẤT CẢ templates trong mỗi cluster
for cluster_id in range(n_clusters):
    cluster_templates = [template_ids[i] for i in range(len(template_ids)) if cluster_labels[i] == cluster_id]
    print(f"\nCluster {cluster_id + 1} ({len(cluster_templates)} templates):")
    
    # Hiển thị HET templates (không giới hạn)
    for tid in cluster_templates:
        idx = template_ids.index(tid)
        template_preview = templates[idx][:100] + "..." if len(templates[idx]) > 100 else templates[idx]
        print(f"  - {tid}: {template_preview}")


PHASE 3: TF-IDF EMBEDDING & TEMPLATE CLUSTERING

Bước 1: Tạo TF-IDF vectors cho templates...
  - Tổng số templates: 820
  - TF-IDF matrix shape: (820, 3277)
  - Vocabulary size: 3277

Bước 2: Tính cosine similarity matrix...
  - Similarity matrix shape: (820, 820)
  - Mean similarity: 0.0469
  - Max similarity (non-diagonal): 0.9899

Top 5 cặp templates giống nhau nhất:
  T-326 <-> T-637: 0.9899
  T-169 <-> T-510: 0.9897
  T-100 <-> T-518: 0.9893
  T-236 <-> T-514: 0.9861
  T-234 <-> T-511: 0.9855

Bước 3: Clustering templates...
  - Số clusters: 5

Cluster 1 (142 templates):
  - T-048: 081109 213908 2549 INFO dfs.DataNode$DataXceiver: 10.251.39.192:50010 Served block blk_-534199272975...
  - T-050: 081109 214043 2561 WARN dfs.DataNode$DataXceiver: 10.251.30.85:50010:Got exception while serving blk...
  - T-051: 081109 214402 2677 WARN dfs.DataNode$DataXceiver: 10.251.126.255:50010:Got exception while serving b...
  - T-053: 081109 214529 2747 WARN dfs.DataNode$DataXceiver: 10.251.123.

In [35]:
# ==========================================
# INJECT LOG LẠ VÀO FILE & DETECT NEW TEMPLATE
# ==========================================

print("="*80)
print("INJECT LOG LẠ VÀO FILE & DETECT NEW TEMPLATE")
print("="*80)

# Lưu số templates hiện tại
old_template_count = len(miner.drain.clusters)
print(f"Số templates hiện có: {old_template_count}\n")

# Tạo 1 dòng log lạ (format JSON - khác hoàn toàn với HDFS format)
strange_log = '{"timestamp":"2008-11-09T21:00:00","level":"FATAL","service":"auth","message":"Database connection pool exhausted - emergency shutdown initiated"}'

# Ghi thêm vào file HDFS_2k.log
print("Đang inject log lạ vào file HDFS_2k.log...")
with open('HDFS_2k.log', 'a', encoding='utf-8') as f:
    f.write('\n' + strange_log)

print(f"✓ Đã ghi log lạ:\n  → {strange_log}\n")

# Parse dòng log mới bằng Drain3
print("Đang parse dòng log mới...")
result = miner.add_log_message(strange_log)

# Kiểm tra kết quả
new_template_count = len(miner.drain.clusters)

print("\n" + "="*80)
print("KẾT QUẢ:")
print(f"  - Templates trước: {old_template_count}")
print(f"  - Templates sau: {new_template_count}")

if new_template_count > old_template_count:
    print(f"\n✓ PHÁT HIỆN TEMPLATE MỚI!")
    print(f"  - Template ID: T-{result['cluster_id']:03d}")
    
    # Lấy template từ cluster
    for cluster in miner.drain.clusters:
        if cluster.cluster_id == result['cluster_id']:
            print(f"  - Template: {cluster.get_template()}")
            print(f"  - Cluster size: {cluster.size}")
            break
else:
    print(f"\n✗ Không tạo template mới")
    print(f"  - Matched với template: T-{result['cluster_id']:03d}")

print("="*80)


INJECT LOG LẠ VÀO FILE & DETECT NEW TEMPLATE
Số templates hiện có: 823

Đang inject log lạ vào file HDFS_2k.log...
✓ Đã ghi log lạ:
  → {"timestamp":"2008-11-09T21:00:00","level":"FATAL","service":"auth","message":"Database connection pool exhausted - emergency shutdown initiated"}

Đang parse dòng log mới...

KẾT QUẢ:
  - Templates trước: 823
  - Templates sau: 823

✗ Không tạo template mới
  - Matched với template: T-823
